# Item 15c — Modo 2 dos modelos de atenção

Terceiro notebook da Phase 7 ([spec](../.specs/features/phase-7-atencao/spec.md), ATT-04, AC5).
Entrada: os 12 checkpoints de `item15b_treinos_atencao_colab.ipynb` e as origens do item 09.
Saída: um `mode2_*.parquet` por (variante, regime, seed, leitura de orçamento), com o **mesmo
esquema de colunas** dos `mode2_*.parquet` de `item09_avaliacao_matriz_colab.ipynb`, e
`variante` valendo `atencao` ou `atencao_emb`.

O protocolo é o do item 09: previsão genuína k passos à frente (`k ∈ {1, 2, 4}`), janela ancorada
na origem, os dois charts, cada predição marcada como on-chart ou piso
([ADR-0004](../docs/adr/0004-recorte-on-chart-como-leitura-principal.md)).

**Origens.** Exatamente as de `origens_{regime}.parquet` do item 09: o veredito do item 15d
compara célula a célula com o sem_grafo e a proposta SAGE. O notebook confere a contagem
(141.504 predições no `current` e 53.982 no `pre_pandemia` por modelo, somando os horizontes) e,
se os parquets do item 09 estiverem no Drive, as próprias chaves.

**As duas leituras de orçamento de arestas** ([ADR-0006](../docs/adr/0006-precompromisso-fase-atencao.md)):
`completo` (grafo inteiro, a **principal**, comparável ao item 10) e `teto_treino` (o mesmo teto de
arestas de cotrajetória do treino, lido do checkpoint). Aqui a `teto_treino` roda em **todas** as
seeds; o item 09 a rodou só na seed 42.

**Runtime: GPU (T4).** O notebook aborta logo depois do setup se não houver GPU. Cada modelo é
reconstruído só a partir dos metadados do checkpoint (`model_from_checkpoint`), sem constantes de
configuração repetidas. Tudo é retomável: cada combinação grava seu parquet e o laço pula os
prontos.

## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.

In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")

In [ ]:
# Portão de GPU: este notebook treina/avalia modelos e só roda numa GPU do Colab (T4).
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Sem GPU: este notebook precisa de um runtime com GPU (T4). No Colab: Ambiente de "
        "execução → Alterar tipo de runtime → GPU, e rode de novo. Abortado antes de qualquer "
        "treino ou avaliação.")
DEVICE = "cuda"
print("DEVICE =", DEVICE, "|", torch.cuda.get_device_name(0))

In [ ]:
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from music_diffusion_gnn.evaluation.longhits import onchart_weeks
from music_diffusion_gnn.evaluation.rollout import gnn_rollout_recursive
from music_diffusion_gnn.evaluation.stats import aggregate_seeds
from music_diffusion_gnn.graph.build import graph_path
from music_diffusion_gnn.graph.controls import rewire_preserving_degree
from music_diffusion_gnn.training.dataset import aggregate_weekly, build_pop_bank
from music_diffusion_gnn.training.trainer import model_from_checkpoint

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
NMAP_PATH = GRAPH_DIR / "node_id_map.json"
TS_PATH   = ROOT / "data" / "processed" / "timeseries.parquet"
print("ROOT =", ROOT, "| DEVICE =", DEVICE)

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/music-influence-gnn")
else:
    BASE = ROOT / "results"
CKPT_DIR = BASE / "atencao" / "treinos"
MATRIZ   = BASE / "avaliacao_matriz"           # item 09: origens e predições da escada SAGE
OUT      = BASE / "atencao" / "avaliacao"
OUT.mkdir(parents=True, exist_ok=True)
print("checkpoints ←", CKPT_DIR, f"({len(list(CKPT_DIR.glob('gnn_*.pt')))} arquivos)")
print("origens     ←", MATRIZ)
print("artefatos   →", OUT)

## 1. Configuração da avaliação

Só o protocolo de avaliação é fixado aqui (horizontes, plano, leituras de orçamento). A
configuração dos modelos vem de cada checkpoint. A ordem do plano roda a leitura principal
(`completo`) de todos os modelos antes da `teto_treino`.

Se a memória da GPU não comportar o grafo completo, a combinação é refeita em lotes de semanas de
origem (`LOTE_SEMANAS`), sem mudar o orçamento de arestas, e a mudança fica registrada em
`avaliacao_log.jsonl` (edge case da spec).

In [ ]:
MAX_WEEK   = 260
KS         = (1, 2, 4)
SEEDS      = [42, 43, 44]
REGIMES    = ["current", "pre_pandemia"]
VARIANTES  = ["atencao", "atencao_emb"]
ORCAMENTOS = ["completo", "teto_treino"]     # completo = principal; teto_treino em todas as seeds
ESPERADO   = {"current": 141_504, "pre_pandemia": 53_982}   # predições por modelo (T11)
LOTE_SEMANAS = 8                              # só se a GPU não comportar o grafo completo
TEMPO_LIMITE_MIN = 150

PLANO = [(orc, v, r, s) for orc in ORCAMENTOS for r in REGIMES for s in SEEDS for v in VARIANTES]
print(f"{len(PLANO)} combinações: {len(VARIANTES)} variantes × {len(REGIMES)} regimes × "
      f"{len(SEEDS)} seeds × {len(ORCAMENTOS)} leituras de orçamento")

## 2. Dados, origens e o recorte on-chart

`onchart_weeks` é a mesma definição do item 09. As origens **não** são recalculadas: são lidas de
`origens_{regime}.parquet` do item 09, e o notebook para se o arquivo não existir.

In [ ]:
ts = pd.read_parquet(TS_PATH)
weekly = aggregate_weekly(ts)
onchart = onchart_weeks(ts, max_week=MAX_WEEK)
print(f"pares (música, chart, semana) on-chart: {len(onchart):,}")

def origens_regime(regime: str) -> pd.DataFrame:
    p = MATRIZ / f"origens_{regime}.parquet"
    if not p.exists():
        raise RuntimeError(f"{p} não existe. As origens têm de ser as do item 09 "
                           "(item09_avaliacao_matriz_colab.ipynb); não as recalcule aqui.")
    return pd.read_parquet(p)

for regime in REGIMES:
    o = origens_regime(regime)
    print(f"{regime}: {len(o):,} origens × {len(KS)} horizontes = {len(o) * len(KS):,} predições "
          f"(esperado {ESPERADO[regime]:,}) | {o.song_id.nunique():,} músicas")
    if len(o) * len(KS) != ESPERADO[regime]:
        print("  ⚠️ a contagem não bate antes do rollout; a conferência depois do rollout decide")

def chaves_item09(regime: str):
    """Chaves (música, chart, origem, k) do Modo 2 da proposta SAGE no item 09, se estiver no Drive."""
    p = MATRIZ / f"mode2_proposta_{regime}_seed42_completo.parquet"
    if not p.exists():
        return None
    ref = pd.read_parquet(p, columns=["song_id", "chart", "origin_week", "k"])
    return set(map(tuple, ref.to_numpy()))

## 3. Carregar um modelo e o grafo que ele viu

`model_from_checkpoint` reconstrói o modelo só com os campos gravados por `checkpoint_dict`
(encoder, `heads`, `attn_dropout`, `hidden`, `layers`, `dropout` e o banco de popularidade do
próprio checkpoint). O `A-emb` é avaliado sobre o **mesmo** grafo embaralhado com que treinou,
refeito a partir de `graph_seed`. A cabeça temporal nasce zerada: se continuar zerada, o
checkpoint não carregou (a assinatura do instrumento quebrado do item 04), e o notebook para.

In [ ]:
_cache_g: dict[str, object] = {}

def grafo(regime: str):
    if regime not in _cache_g:
        _cache_g[regime] = torch.load(graph_path(regime, GRAPH_DIR), weights_only=False)
    return _cache_g[regime]

def grafo_da_variante(g, variante: str, graph_seed):
    if variante == "atencao":
        return g
    if variante == "atencao_emb":
        assert graph_seed is not None, "checkpoint atencao_emb sem graph_seed: topologia irrecuperável"
        return rewire_preserving_degree(g, seed=int(graph_seed))
    raise ValueError(variante)

def carregar(variante: str, regime: str, seed: int):
    """Devolve (modelo pronto para eval, grafo da variante, checkpoint) ou (None, None, None)."""
    ck_path = CKPT_DIR / f"gnn_{variante}_{regime}_seed{seed}.pt"
    if not ck_path.exists():
        return None, None, None
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    assert (ck["variante"], ck["split_regime"], ck["seed"]) == (variante, regime, seed), ck_path
    g = grafo(regime)
    modelo = model_from_checkpoint(ck, g.metadata())
    assert modelo.pop_bank is not None, f"{ck_path.name}: checkpoint sem pop_bank"
    assert modelo.head.mlp[-1].weight.abs().max().item() > 0, (
        f"{ck_path.name}: cabeça temporal zerada, o checkpoint não foi carregado")
    g_var = grafo_da_variante(g, ck["variante"], ck["graph_seed"])
    return modelo.to(DEVICE).eval(), g_var, ck

_m, _g, _ck = carregar("atencao", "current", 42)
print("checkpoint de referência:",
      f"{_ck['config_str']} | {_ck['n_params']:,} parâmetros | {_ck['graph_version']}"
      if _ck else "AUSENTE — rode o item 15b primeiro")

## 4. Calibração — quanto custa o Modo 2 da atenção

Roda 200 origens na leitura `completo` e extrapola para o plano inteiro.

In [ ]:
if _m is not None:
    amostra = origens_regime("current").head(200)
    t0 = time.time()
    _ = gnn_rollout_recursive(_m, _g, weekly, amostra, W=_ck["W"], ks=KS, device=DEVICE,
                              regime="current", max_cotraj_edges=None)
    por_origem = (time.time() - t0) / len(amostra)
    n_total = sum(len(origens_regime(r)) for r in REGIMES) * len(SEEDS) * len(VARIANTES) * len(ORCAMENTOS)
    print(f"{por_origem*1000:.1f} ms/origem → plano completo ≈ {por_origem*n_total/3600:.1f} h de GPU")
    del _m, _g
    torch.cuda.empty_cache()

## 5. Modo 2 — as predições

Um parquet por combinação, com as colunas do item 09: `song_id`, `chart`, `origin_week`, `k`,
`y_true`, `y_pred`, `target_week`, `onchart`, `variante`, `split_regime`, `seed`,
`orcamento_arestas` e `max_cotraj_edges` (−1 no grafo completo). Antes de gravar, a contagem é
conferida contra o esperado e, quando possível, as chaves contra as do item 09; se não baterem,
nada é gravado.

In [ ]:
def marcar_recorte(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["target_week"] = df["origin_week"] + df["k"]
    df["onchart"] = [
        (sid, chart, int(w)) in onchart
        for sid, chart, w in zip(df["song_id"], df["chart"], df["target_week"])
    ]
    return df

def registrar(evento: dict):
    with open(OUT / "avaliacao_log.jsonl", "a") as f:
        f.write(json.dumps(evento | {"quando": time.strftime("%Y-%m-%d %H:%M:%S")}) + "\n")

def rollout(modelo, g_var, ck, origens, regime, orcamento):
    kw = dict(W=ck["W"], ks=KS, device=DEVICE, regime=regime, max_cotraj_edges=orcamento)
    try:
        return gnn_rollout_recursive(modelo, g_var, weekly, origens, **kw), False
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        semanas = sorted(origens["week"].unique())
        lotes = [semanas[i:i + LOTE_SEMANAS] for i in range(0, len(semanas), LOTE_SEMANAS)]
        print(f"  memória da GPU insuficiente: refazendo em {len(lotes)} lotes de semanas")
        partes = []
        for lote in lotes:
            partes.append(gnn_rollout_recursive(modelo, g_var, weekly,
                                                origens[origens["week"].isin(lote)], **kw))
            torch.cuda.empty_cache()
        return pd.concat(partes, ignore_index=True), True

t_inicio = time.time()
for nome_orc, variante, regime, seed in PLANO:
    alvo = OUT / f"mode2_{variante}_{regime}_seed{seed}_{nome_orc}.parquet"
    if alvo.exists():
        print(f"{alvo.name}: pronto — pulando")
        continue
    if (time.time() - t_inicio) / 60 > TEMPO_LIMITE_MIN:
        print(f"\nlimite de {TEMPO_LIMITE_MIN} min atingido. Rode a célula de novo numa sessão "
              "nova: o que já saiu será pulado.")
        break

    modelo, g_var, ck = carregar(variante, regime, seed)
    if modelo is None:
        print(f"{variante}/{regime}/seed{seed}: checkpoint ausente — pulando")
        continue
    orcamento = None if nome_orc == "completo" else int(ck["max_cotraj_edges"])
    t0 = time.time()
    df, em_lotes = rollout(modelo, g_var, ck, origens_regime(regime), regime, orcamento)
    if len(df) != ESPERADO[regime]:
        raise RuntimeError(f"{alvo.name}: {len(df):,} predições, esperado {ESPERADO[regime]:,}. "
                           "As origens não são as do item 09; nada foi gravado.")
    ref = chaves_item09(regime)
    if ref is not None and set(map(tuple, df[["song_id", "chart", "origin_week", "k"]].to_numpy())) != ref:
        raise RuntimeError(f"{alvo.name}: as chaves (música, chart, origem, k) diferem das do "
                           "item 09; nada foi gravado.")
    df = marcar_recorte(df)
    df["variante"], df["split_regime"], df["seed"] = variante, regime, seed
    df["orcamento_arestas"] = nome_orc
    df["max_cotraj_edges"] = -1 if orcamento is None else orcamento
    df.to_parquet(alvo, index=False)
    registrar({"arquivo": alvo.name, "linhas": len(df), "em_lotes_de_semanas": em_lotes,
               "lote_semanas": LOTE_SEMANAS if em_lotes else None,
               "minutos": (time.time() - t0) / 60, "graph_version": ck["graph_version"]})
    print(f"{alvo.name}: {len(df):,} linhas, on-chart {df.onchart.mean():.1%} "
          f"[{(time.time()-t0)/60:.1f} min]" + (" (em lotes de semanas)" if em_lotes else ""))
    del modelo, g_var
    torch.cuda.empty_cache()

## 6. Conferência e leitura rápida

Primeiro, o esquema e as contagens: cada parquet com as colunas exigidas, `variante` só
`atencao`/`atencao_emb`, e a contagem por modelo igual à esperada. Depois o RMSE por célula, com
média e desvio entre seeds, nas duas leituras de orçamento. Ainda não é o veredito: falta a
estatística pareada do item 15d.

In [ ]:
COLUNAS = ["variante", "split_regime", "seed", "song_id", "chart", "origin_week", "k",
           "y_true", "y_pred", "onchart", "orcamento_arestas"]
arquivos = sorted(OUT.glob("mode2_*.parquet"))
if not arquivos:
    raise SystemExit("nenhum parquet de Modo 2 ainda: rode a célula anterior")
m2 = pd.concat([pd.read_parquet(f) for f in arquivos], ignore_index=True)
faltam_cols = [c for c in COLUNAS if c not in m2.columns]
assert not faltam_cols, f"colunas faltando: {faltam_cols}"
assert set(m2.variante) <= {"atencao", "atencao_emb"}, set(m2.variante)
contagem = m2.groupby(["variante", "split_regime", "seed", "orcamento_arestas"]).size()
assert all(n == ESPERADO[idx[1]] for idx, n in contagem.items()), contagem
display(contagem.unstack("orcamento_arestas"))

prontos = {(o, v, r, s) for o, v, r, s in PLANO
           if (OUT / f"mode2_{v}_{r}_seed{s}_{o}.parquet").exists()}
print(f"{len(prontos)} de {len(PLANO)} combinações prontas; teto_treino em "
      f"{sorted({s for o, v, r, s in prontos if o == 'teto_treino'})} "
      f"(das seeds {SEEDS})")

m2 = m2.dropna(subset=["y_true", "y_pred"])
CHAVE = ["variante", "split_regime", "orcamento_arestas", "chart", "k"]

def rmse_por_seed(df, recorte):
    sub = df if recorte == "full" else df[df.onchart]
    e = sub.assign(err2=(sub.y_true - sub.y_pred) ** 2)
    out = e.groupby(CHAVE + ["seed"])["err2"].mean().pow(0.5).reset_index(name="rmse")
    out["recorte"] = recorte
    return out

por_seed = pd.concat([rmse_por_seed(m2, r) for r in ("full", "onchart")], ignore_index=True)
linhas = []
for chave, grp in por_seed.groupby(CHAVE + ["recorte"]):
    media, desvio = aggregate_seeds(grp["rmse"].to_numpy())
    linhas.append(dict(zip(CHAVE + ["recorte"], chave))
                  | {"rmse_medio": media, "rmse_desvio": desvio, "n_seeds": len(grp)})
resumo = pd.DataFrame(linhas)
resumo.to_parquet(OUT / "resumo_mode2_atencao.parquet", index=False)

for orc in ORCAMENTOS:
    sub = resumo[(resumo.orcamento_arestas == orc) & (resumo.recorte == "onchart")]
    if len(sub):
        print(f"\nleitura {orc}, on-chart:")
        display(sub.pivot_table(index=["split_regime", "chart", "k"], columns="variante",
                                values="rmse_medio").round(6))

## 7. O que levar de volta

Do Drive (`atencao/avaliacao/`):

- `mode2_{variante}_{regime}_seed{seed}_{completo|teto_treino}.parquet` — as predições, já marcadas
  por recorte, com o esquema do item 09;
- `resumo_mode2_atencao.parquet` — RMSE médio e desvio entre seeds por célula;
- `avaliacao_log.jsonl` — tempo de cada combinação e se ela precisou de lotes de semanas.

Em seguida, `item15d_veredito_atencao_colab.ipynb`.